# Park (2016)

Append Deborah's requested annual maximum-temperature bins and mean daily precipitation to existing Illinois county NetCDF files. Keep this notebook beside `loca2_common.py` and run from top to bottom.

Only missing variables are added, using netCDF4 append mode. Existing variables, coordinates, attributes, filenames, and directory paths are preserved. No new climate output files are created. Do not run another writer on the same files while this notebook runs.

County aggregation uses the existing cosine-latitude area weighting, with no population, LandScan, or crop weights. Input and output roots remain configured in `loca2_common.py`.

The 12 exclusive bins span below 0°F through ≥95°F. The existing ≥90°F variable is retained but must not be included alongside the 90–<95°F and ≥95°F bins when summing all bins. County counts can be fractional because grid-cell day counts are averaged spatially.

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [2]:
from pathlib import Path
import calendar
import numpy as np
import xarray as xr
import netCDF4
from tqdm.auto import tqdm
import loca2_common as common
from loca2_common import aggregate_to_illinois_counties, to_fahrenheit

In [7]:
# Exclude temporal summaries from daily input discovery.
# Preserve the original function so this cell can be rerun safely.
if not hasattr(common, "_park_original_nc_files"):
    common._park_original_nc_files = common.nc_files


def park_daily_nc_files(directory):
    files = common._park_original_nc_files(directory)

    excluded_markers = (
        ".yearly.",
        ".annual.",
        ".monthly.",
        ".seasonal.",
        ".us_county_avg.",
    )

    daily_files = [
        filename
        for filename in files
        if not any(
            marker in Path(filename).name.lower()
            for marker in excluded_markers
        )
    ]

    return daily_files


common.nc_files = park_daily_nc_files

# Verify the historical precipitation files now selected.
historical_folder = common.historical_dir(
    common.PAPER_INPUT_ROOT,
    "pr",
    "ACCESS-CM2",
    "r1i1p1f1",
)

print("Selected daily historical precipitation files:")
for filename in common.nc_files(historical_folder):
    print(Path(filename).name)

Selected daily historical precipitation files:
pr.ACCESS-CM2.historical.r1i1p1f1.1950-2014.LOCA_16thdeg_v20240915.cent.Illinois.nc


## 2. Paper settings

In [8]:
PAPER_NAME = 'Park (2016)'
REQUIRED_VARIABLES = ('tasmax', 'pr')
OVERWRITE = False  # Existing variables are never overwritten.

BIN_SPECS = [('tmax_days_lt0F', -np.inf, 0.0)] + [
    (f'tmax_days_{lo}_{lo + 10}F', float(lo), float(lo + 10))
    for lo in range(0, 90, 10)
] + [('tmax_days_90_95F', 90.0, 95.0),
     ('tmax_days_ge95F', 95.0, np.inf)]
PRECIP_NAME = 'annual_mean_daily_precipitation_tenths_inch'
REQUESTED_VARIABLES = [name for name, _, _ in BIN_SPECS] + [PRECIP_NAME]

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [9]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
files = sorted(output_directory.glob('*.nc'))
if not files:
    raise FileNotFoundError(f'No existing NetCDF files found: {output_directory}')

jobs = []
for path in files:
    with xr.open_dataset(path, decode_timedelta=False) as existing:
        missing = [name for name in REQUESTED_VARIABLES if name not in existing]
        if not missing:
            print('COMPLETE:', path.name)
            continue
        required_attrs = ('model', 'member', 'scenario')
        if any(key not in existing.attrs for key in required_attrs):
            raise ValueError(f'{path.name}: missing model/member/scenario attributes')
        jobs.append((path, *(existing.attrs[key] for key in required_attrs)))
print('Input root:', common.PAPER_INPUT_ROOT)
print('Existing output folder:', output_directory)
print('Existing files:', len(files))
print('Files needing additions:', len(jobs))
counties = common.load_counties()

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Existing output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Park (2016)
Existing files: 156
Files needing additions: 156


## 4. Paper indicator calculation

In [10]:
def select_complete_years(data, years):
    data = data.where(
        data.time.dt.year.isin(years),
        drop=True,
    ).sortby("time")

    dates = np.asarray(
        data.time.dt.strftime("%Y-%m-%d").values
    )

    unique_dates, first_indices, counts = np.unique(
        dates,
        return_index=True,
        return_counts=True,
    )

    duplicate_dates = unique_dates[counts > 1]

    if duplicate_dates.size:
        print(
            f"{data.name}: checking "
            f"{duplicate_dates.size} duplicated calendar dates."
        )

        for date in duplicate_dates:
            positions = np.flatnonzero(dates == date)

            reference = data.isel(
                time=int(positions[0]),
                drop=True,
            )

            for position in positions[1:]:
                candidate = data.isel(
                    time=int(position),
                    drop=True,
                )

                # Require exact agreement, treating paired NaNs as equal.
                matches = (
                    (reference == candidate)
                    | (reference.isnull() & candidate.isnull())
                ).all()

                if not bool(matches.compute().item()):
                    raise ValueError(
                        f"{data.name}: conflicting records for {date}. "
                        "Inspect the source files before choosing a record."
                    )

        removed = len(dates) - len(unique_dates)

        data = data.isel(
            time=np.sort(first_indices)
        )

        print(
            f"{data.name}: removed {removed} matching duplicate "
            "records; retained one record per calendar date."
        )

    calendar_name = data.time.dt.calendar
    observed_years = data.time.dt.year.values

    for year in years:
        year = int(year)

        if calendar_name == "360_day":
            expected = 360
        elif calendar_name in ("noleap", "365_day"):
            expected = 365
        elif calendar_name in ("all_leap", "366_day"):
            expected = 366
        elif calendar_name == "julian":
            expected = 366 if year % 4 == 0 else 365
        else:
            expected = 366 if calendar.isleap(year) else 365

        count = int(
            np.count_nonzero(observed_years == year)
        )

        if count != expected:
            raise ValueError(
                f"{data.name}: year {year} has "
                f"{count}/{expected} unique daily records."
            )

    return data


def calculate_park_additions(inputs, years, missing):
    variables = {}
    temperature_names = {name for name, _, _ in BIN_SPECS}
    if temperature_names.intersection(missing):
        tmax = to_fahrenheit(select_complete_years(inputs['tasmax'], years))
        # A missing daily temperature invalidates the grid-cell annual count.
        valid = np.isfinite(tmax).groupby('time.year').all('time')
        for name, lower, upper in BIN_SPECS:
            if name not in missing:
                continue
            count = ((tmax >= lower) & (tmax < upper)).groupby('time.year').sum('time')
            variables[name] = count.where(valid)
            variables[name].attrs.update(
                units='days', frequency='annual',
                temperature_statistic='daily maximum temperature',
                long_name=f'annual days with {lower:g} <= Tmax < {upper:g} degrees Fahrenheit',
                spatial_interpretation='county-area-weighted average of grid-cell day counts',
            )
            if np.isfinite(lower):
                variables[name].attrs['lower_threshold_fahrenheit'] = lower
            if np.isfinite(upper):
                variables[name].attrs['upper_threshold_fahrenheit'] = upper

    if PRECIP_NAME in missing:
        pr = select_complete_years(inputs['pr'], years)
        units = str(pr.attrs.get('units', '')).strip().lower().replace('−', '-')
        accepted = {'inch', 'inches', 'in', 'inch/day', 'inches/day', 'in/day',
                    'inch day-1', 'inches day-1', 'in day-1',
                    'inch d-1', 'inches d-1', 'in d-1'}
        if units not in accepted:
            raise ValueError(f'Expected daily precipitation in inches; found {units!r}')
        variables[PRECIP_NAME] = pr.where(np.isfinite(pr)).groupby('time.year').mean(
            'time', skipna=False
        ) * 10.0
        variables[PRECIP_NAME].attrs.update(
            units='0.1 inch day-1', frequency='annual',
            long_name='annual mean daily precipitation in tenths of an inch per day',
            calculation='10 * mean(daily precipitation in inches) over each calendar year',
        )
    # Aggregate independently: tasmax and precipitation need not share timestamps.
    pieces = []
    for name, data in variables.items():
        attrs = dict(data.attrs)
        county_data = aggregate_to_illinois_counties(
            dataset=xr.Dataset({name: data}), counties=counties
        )
        county_data[name].attrs.update(attrs)
        pieces.append(county_data)
    return xr.merge(pieces, join='exact', compat='equals')


def match_existing_layout(annual, existing):
    if 'calendar_year' not in existing:
        raise ValueError('Existing output has no calendar_year variable')
    for coordinate in ('county', 'county_id'):
        if coordinate not in existing.coords or coordinate not in annual.coords:
            raise ValueError(f'Missing coordinate: {coordinate}')
        np.testing.assert_array_equal(existing[coordinate].values, annual[coordinate].values)
    # Select by actual years, preserving existing period names and year_index values.
    indexer = existing['calendar_year'].astype('int64')
    additions = annual.sel(year=indexer)
    if 'year' in additions.coords:
        additions = additions.drop_vars('year')
    for name in additions.data_vars:
        additions[name] = additions[name].transpose(*indexer.dims, 'county')
        for dim in additions[name].dims:
            if additions.sizes[dim] != existing.sizes[dim]:
                raise ValueError(f'Dimension mismatch: {dim}')
            if dim in existing.coords:
                np.testing.assert_array_equal(additions[dim].values, existing[dim].values)
    return additions


def append_missing_variables(path, additions):
    # Complete every calculation before opening the existing file for modification.
    additions.load()
    with netCDF4.Dataset(path, mode='a') as target:
        for name, data in additions.data_vars.items():
            if name in target.variables:
                raise ValueError(f'{name} was added by another process; stopping')
            for dim in data.dims:
                if dim not in target.dimensions or len(target.dimensions[dim]) != data.sizes[dim]:
                    raise ValueError(f'{name}: incompatible dimension {dim}')
        for name, data in additions.data_vars.items():
            variable = target.createVariable(
                name, 'f8', data.dims, zlib=True, complevel=4, shuffle=True,
                fill_value=np.nan,
            )
            variable.setncatts(dict(data.attrs))
            variable[:] = data.values
        target.sync()

## 5. Append missing variables to existing files

No call to `common.run_paper` is made because it would skip or replace whole files. This cell loads the required daily inputs, matches the existing calendar-year and county coordinates, and adds missing variables using netCDF4 append mode. Existing variables are skipped on reruns.

Incomplete daily time coverage raises an error. Missing daily values invalidate that grid-cell year; the shared county function averages available grid cells. No temporary climate files or replacement output files are written. In-place appends are not transactional: an interrupted write must be checked before rerunning.

In [11]:
updated = skipped = failed = 0
for path, model, member, scenario in tqdm(jobs, desc='Append Park variables', unit='file'):
    inputs = {}
    additions = None
    try:
        with xr.open_dataset(path, decode_timedelta=False) as existing:
            missing = [name for name in REQUESTED_VARIABLES if name not in existing]
            if not missing:
                skipped += 1
                continue
            years = np.unique(existing['calendar_year'].values).astype(int)
            required = []
            if any(name != PRECIP_NAME for name in missing):
                required.append('tasmax')
            if PRECIP_NAME in missing:
                required.append('pr')
            inputs = common.load_paper_inputs(model, member, scenario, tuple(required))
            annual = calculate_park_additions(inputs, years, missing)
            additions = match_existing_layout(annual, existing).load()
        # The read handle is closed before opening in append mode.
        append_missing_variables(path, additions)
        updated += 1
        print(f'UPDATED {path.name}: added {", ".join(missing)}')
    except Exception as exc:
        failed += 1
        print(f'FAILED {path.name}: {type(exc).__name__}: {exc}')
    finally:
        if additions is not None:
            additions.close()
        for data in inputs.values():
            data.close()
print(f'Updated={updated}, already complete={skipped}, failed={failed}')

Append Park variables:   0%|          | 0/156 [00:00<?, ?file/s]

UPDATED Park_2016__ACCESS-CM2_r1i1p1f1_ssp245.nc: added tmax_days_lt0F, tmax_days_0_10F, tmax_days_10_20F, tmax_days_20_30F, tmax_days_30_40F, tmax_days_40_50F, tmax_days_50_60F, tmax_days_60_70F, tmax_days_70_80F, tmax_days_90_95F, tmax_days_ge95F, annual_mean_daily_precipitation_tenths_inch
UPDATED Park_2016__ACCESS-CM2_r1i1p1f1_ssp370.nc: added tmax_days_lt0F, tmax_days_0_10F, tmax_days_10_20F, tmax_days_20_30F, tmax_days_30_40F, tmax_days_40_50F, tmax_days_50_60F, tmax_days_60_70F, tmax_days_70_80F, tmax_days_90_95F, tmax_days_ge95F, annual_mean_daily_precipitation_tenths_inch
UPDATED Park_2016__ACCESS-CM2_r2i1p1f1_ssp245.nc: added tmax_days_lt0F, tmax_days_0_10F, tmax_days_10_20F, tmax_days_20_30F, tmax_days_30_40F, tmax_days_40_50F, tmax_days_50_60F, tmax_days_60_70F, tmax_days_70_80F, tmax_days_90_95F, tmax_days_ge95F, annual_mean_daily_precipitation_tenths_inch
UPDATED Park_2016__ACCESS-CM2_r2i1p1f1_ssp370.nc: added tmax_days_lt0F, tmax_days_0_10F, tmax_days_10_20F, tmax_days_2

## 6. Inspect an output file

In [12]:
for path in files:
    with xr.open_dataset(path, decode_timedelta=False) as dataset:
        missing = [name for name in REQUESTED_VARIABLES if name not in dataset]
        if missing:
            print('MISSING:', path.name, missing)
            continue
        bins = dataset[[name for name, _, _ in BIN_SPECS]].to_array('temperature_bin')
        total = bins.sum('temperature_bin', skipna=False)
        print(path.name)
        print('Dimensions:', dict(dataset.sizes))
        print('Sum of 12 exclusive bins, min/max:', float(total.min()), float(total.max()))
        print('Precipitation units:', dataset[PRECIP_NAME].attrs.get('units'))
        print('Missing bin values:', int(bins.isnull().sum()))
        print('Missing precipitation values:', int(dataset[PRECIP_NAME].isnull().sum()))
        # Complete county-year bin sums should match the source calendar year length.
        # Do not include the retained tmax_days_ge90F in this sum.

Park_2016__ACCESS-CM2_r1i1p1f1_ssp245.nc
Dimensions: {'period': 4, 'year_index': 30, 'county': 102}
Sum of 12 exclusive bins, min/max: 364.99999999999983 366.00000000000017
Precipitation units: 0.1 inch day-1
Missing bin values: 0
Missing precipitation values: 0
Park_2016__ACCESS-CM2_r1i1p1f1_ssp370.nc
Dimensions: {'period': 4, 'year_index': 30, 'county': 102}
Sum of 12 exclusive bins, min/max: 364.99999999999983 366.00000000000017
Precipitation units: 0.1 inch day-1
Missing bin values: 0
Missing precipitation values: 0
Park_2016__ACCESS-CM2_r2i1p1f1_ssp245.nc
Dimensions: {'period': 4, 'year_index': 30, 'county': 102}
Sum of 12 exclusive bins, min/max: 364.99999999999983 366.00000000000017
Precipitation units: 0.1 inch day-1
Missing bin values: 0
Missing precipitation values: 0
Park_2016__ACCESS-CM2_r2i1p1f1_ssp370.nc
Dimensions: {'period': 4, 'year_index': 30, 'county': 102}
Sum of 12 exclusive bins, min/max: 364.99999999999983 366.00000000000017
Precipitation units: 0.1 inch day-1
Mi

In [13]:
from pathlib import Path
import numpy as np
import xarray as xr

MODEL = "ACCESS-CM2"
MEMBER = "r1i1p1f1"
CHECK_DATE = "1985-07-01"

historical_folder = common.historical_dir(
    common.PAPER_INPUT_ROOT,
    "pr",
    MODEL,
    MEMBER,
)

source_files = common.nc_files(historical_folder)

print("Historical folder:", historical_folder)
print("Source files:", len(source_files))

for filename in source_files:
    with xr.open_dataset(filename) as ds:
        if "time" not in ds.coords:
            continue

        dates = np.asarray(
            ds.time.dt.strftime("%Y-%m-%d").values
        )
        positions = np.flatnonzero(dates == CHECK_DATE)

        if not len(positions):
            continue

        print("\nFILE:", Path(filename).name)
        print("Matching records:", len(positions))

        # Inspect precipitation variables without assuming a variable name.
        variable_names = [
            name
            for name, variable in ds.data_vars.items()
            if "time" in variable.dims
            and (
                name.lower().startswith("pr")
                or "precip" in name.lower()
                or "precip" in str(
                    variable.attrs.get("standard_name", "")
                ).lower()
            )
        ]

        if not variable_names:
            print("Available variables:", list(ds.data_vars))
            continue

        for name in variable_names:
            print("Variable:", name)
            print("Units:", ds[name].attrs.get("units"))

            for position in positions:
                record = ds[name].isel(time=int(position)).load()

                print(
                    "Timestamp:",
                    str(ds.time.values[int(position)]),
                )
                print(
                    "Min / mean / max:",
                    float(record.min(skipna=True)),
                    float(record.mean(skipna=True)),
                    float(record.max(skipna=True)),
                )
                print(
                    "Missing cells:",
                    int(record.isnull().sum()),
                )

Historical folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/pr/historical/ACCESS-CM2/r1i1p1f1
Source files: 1

FILE: pr.ACCESS-CM2.historical.r1i1p1f1.1950-2014.LOCA_16thdeg_v20240915.cent.Illinois.nc
Matching records: 1
Variable: pr
Units: kg m-2 s-1
Timestamp: 1985-07-01T12:00:00.000000000
Min / mean / max: 0.0 0.0 0.0
Missing cells: 144
